## Workspace setup

In [5]:
from datetime import datetime 
import uproot
import awkward as ak
import tensorflow as tf
import numpy as np
import importlib
from functools import partial

from tensorflow.data import Dataset, TFRecordDataset
from tensorflow.data.experimental import TFRecordWriter
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Example, Features, Feature
import tensorflow_datasets as tfds

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

import io_functions as io

### Convert simulated data.

Simulated data contains Track3D objects, for generated and reconstructed tracks.
We create TFRecords for both SimEvent and RecoEvent.

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC_20k.root:TPCData',
             dataPath+'SimEvent_Track3D_TwoProng_gun_MC_50k.root:TPCData'
             ]


simOutputDir = 'SimEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, simOutputDir, fields= io.simEventFields)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
# uproot always takes the first branch with given name, unless 
# explicit branch is given as input path. Filtering by branches in
# iterate does not work. 
rootfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC_20k.root:TPCRecoData/RecoEvent',
             dataPath+'SimEvent_Track3D_TwoProng_gun_MC_50k.root:TPCRecoData/RecoEvent'
             ]
recoOutputDir = 'RecoEvent_Track3D_TwoProng_gun_MC'

# Convert ROOT files to TF format and save to output directory
io.convertROOT(rootfiles, recoOutputDir, fields=io.recoEventFields)

### Merge SimEvent and RecoEvent data


In [ ]:
# merge SimEvent and RecoEvent data
simDataset = tf.data.Dataset.load(simOutputDir, compression="GZIP")
recoDataset = tf.data.Dataset.load(recoOutputDir, compression="GZIP")

mergedDataset = tf.data.Dataset.zip((simDataset, recoDataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)

# filter merged dataset
# calculate sim alpha length and select events with legth > 20 mm
minAlphaLength = 20.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.greater(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), minAlphaLength
        )
    )
)

# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_gun_MC'
mergedDataset.save(outputDir, compression="GZIP")

### Load and convert data events

Real data events contain reconstructed Track3D in both trees: RecoEvent and SimEvent.
We load the RecoEvent and put the data twice into the dict to maintain the same structure as for simulated data.

In [8]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
# uproot always takes the first branch with given name, unless 
# explicit branch is givent as input path. Filtering by branches in
# iterate does not work. 
rootfiles = [dataPath+'RecoEvent_TwoProng_2022-04-12T08-03-44.root:TPCData'
             ]
outputDir = 'RecoEvent_Track3D_TwoProng_2022-04-12T08-03-44'

# Convert ROOT files to TF format and save to output directory
# use the simEventFields which containt the Track3D and the event data (images)
io.convertROOT(rootfiles, outputDir, fields=io.simEventFields)

dataset = tf.data.Dataset.load(outputDir, compression="GZIP")

mergedDataset = tf.data.Dataset.zip((dataset, dataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)

# filter merged dataset
# calculate sim alpha length and select events with legth > 20 mm
minAlphaLength = 20.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.greater(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), minAlphaLength
        )
    )
)

# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_2022-04-12T08-03-44'
mergedDataset.save(outputDir, compression="GZIP")

Cause: could not parse the source code of <function <lambda> at 0x71243ebcaa20>: no matching AST found among candidates:

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: could not parse the source code of <function <lambda> at 0x71243ebcaa20>: no matching AST found among candidates:

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: could not parse the source code of <function <lambda> at 0x712542f12480>: no matching AST found among candidates:

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: could not parse the source code of <function <lambda> at 0x712542f12480>: no matching AST found among candidates:

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
CPU times: user 1.37 s, sys: 216 ms, total: 1.58 s
Wall time: 1.56 s
